# Anthive — Getting Started

Run each cell with **Shift+Enter**. This notebook talks to the anthive REST API from your browser (Pyodide + [`vark`](https://codeberg.org/mfiers/pyvark) — vendored, no install needed for the client itself). It always picks the **first available dataset** on the server, so it works out of the box regardless of what's loaded.

Once you've seen it work, copy any cell into a new notebook and start exploring — swap the dataset id, the gene, the metadata field. Edits are saved to your browser's local storage; download with *File → Download* to keep them permanently.


In [ ]:
# 1. Install requests (vark's HTTP layer) and the two scientific
#    dependencies (vark itself is vendored next to this notebook), and
#    give Pyodide's streaming fetch time to spin up.  Without the warm-up,
#    urllib3's emscripten backend falls back to XHR and truncates JSON
#    responses larger than ~56 KB.
import sys
if 'pyodide' in sys.modules:
    import micropip
    await micropip.install(['requests', 'pandas', 'matplotlib'])
    import asyncio
    await asyncio.sleep(0.5)   # give the streaming service worker a beat

from vark import AnthiveClient
print('dependencies loaded')


In [ ]:
# 2. Connect to the anthive server + pick the FIRST available dataset.
#    In JupyterLite the API URL is auto-detected from js.location.origin;
#    the Authelia/basic-auth cookie the browser already has is reused.
try:
    import js
    api_url = f'{js.location.origin}/api/'
except ImportError:
    api_url = None   # fall back to the ANTHIVE_API_URL env var (local Jupyter)

client = AnthiveClient(api_url)

databases = client.get_databases(format='dataframe')
if len(databases) == 0:
    raise SystemExit('No datasets on the server yet — ingest one first with `ant convert my.h5ad`.')

db_id = databases.iloc[0]['id']
print(f'Server:  {client.base_url}')
print(f'{len(databases)} dataset(s) available; using first: {db_id!r}')
databases[['id', 'title', 'n_cells', 'n_genes']].head()


In [ ]:
# 3. Inspect the chosen dataset + pick a real gene to plot.
#    Layer preference: 'X' (canonical), else the first available.
info = client.get_database_info(db_id)
layers = list(info.get('layers') or ['X'])   # the server sends {key: display name}
layer = 'X' if 'X' in layers else layers[0]
print(f'cells    : {info["n_cells"]:,}')
print(f'genes    : {info["n_genes"]:,}')
print(f'layer    : {layer!r}   (available: {", ".join(layers)})')

# Ask the server for one gene — we take whatever it gives us so this
# notebook is dataset-agnostic. Empty search returns everything.
genes_found = client.search_genes(db_id, q='', limit=5)
if not genes_found:
    raise SystemExit(f'Dataset {db_id!r} reports zero genes — check the ingest.')
gene = genes_found[0]
print(f'gene     : {gene!r}   (chose first of {genes_found})')


In [ ]:
# 4. Fetch the per-cell expression + plot a histogram.
#    Small, self-contained, works on any dataset shape.
import matplotlib.pyplot as plt

cells = client.get_cells(db_id, genes=[gene], layer=layer, format='dataframe')
print(f'fetched  : {len(cells):,} cells x 1 gene')

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(cells[gene], bins=40, color='#4b6cb7', alpha=0.8)
ax.set_xlabel(f'{gene} expression ({layer})')
ax.set_ylabel('cells')
ax.set_title(f'{gene} distribution in {db_id}')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()


## Where to go next

- `client.get_databases()` — list every dataset.
- `client.get_database_info(db_id)` — cells, genes, layers, embeddings, obs fields.
- `client.search_genes(db_id, q='TREM')` — find genes by substring.
- `client.get_cells(db_id, genes=[...], metadata=[...], format='dataframe')` — fetch a per-cell table.
- `client.get_plot(db_id, 'scatter', color='...', bin_color=3)` — server-rendered PNG + caption.

Full docs: [`codeberg.org/mfiers/pyvark`](https://codeberg.org/mfiers/pyvark).
